# Exercise 8: Lake Formation Governance — Genomic Data Access Control
## BINFX410 — Chapter 10

---

## Learning Objectives
- Explain the limitations of S3 bucket policies for fine-grained genomic data governance
- Configure AWS Lake Formation column-level security to mask PHI fields
- Apply row-level filters to restrict researcher access to approved cohorts
- Test access control using IAM role assumption
- Connect Lake Formation governance to HIPAA and dbGaP access frameworks

**Estimated time:** 60 minutes  
**Dataset:** Synthetic patient-linked genomic data (generated in this notebook)  
**AWS services:** Lake Formation, IAM, S3, Glue, Athena

**Note:** This exercise requires Lake Formation to be enabled in your account. Your instructor may need to set you as a Lake Formation data lake administrator.

## Background: The Genomic Data Governance Problem

### Open-Access vs Controlled-Access Data

The genomics community has developed a two-tier access model:

| Tier | Examples | What's Shared |
|---|---|---|
| **Open Access** | GTEx summary, gnomAD population frequencies, ClinVar | Aggregate statistics — no individual-level data |
| **Controlled Access** | TCGA clinical + genomic, dbGaP studies, UK Biobank | Individual-level data requiring IRB approval and data use agreement |

TCGA's controlled-access tier (`s3://tcga-2-controlled`) requires:
1. Approved dbGaP data access request
2. Signed Data Use Agreement
3. NIH eRA Commons credentials

### Why S3 Bucket Policies Are Insufficient

S3 bucket policies control access at the **object level** — either a user can read a file or they can't. They cannot:
- Allow a user to read an object but see only certain columns
- Allow a user to see only rows matching a specific filter
- Log who accessed which specific data elements

**Lake Formation** adds a governance layer above S3 that enables:
- Column-level security (mask PHI columns for unauthorized users)
- Row-level filtering (researcher sees only their approved cohort)
- Data access auditing (CloudTrail integration)
- Centralized policy management across all Athena queries

## Section 1: Setup

In [ ]:
!pip install boto3 awswrangler pandas faker --quiet

In [ ]:
import boto3
import awswrangler as wr
import pandas as pd
import json
from faker import Faker
import random
import numpy as np

STUDENT_ID = "sab032226"  # CHANGE THIS
AWS_REGION = "us-east-1"
BUCKET = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_RESULTS = f"s3://binfx410-athena-results-{STUDENT_ID}/"

s3 = boto3.client('s3', region_name=AWS_REGION)
glue = boto3.client('glue', region_name=AWS_REGION)
lf = boto3.client('lakeformation', region_name=AWS_REGION)
iam = boto3.client('iam')
sts = boto3.client('sts')
session = boto3.Session(region_name=AWS_REGION)

# Get current account ID and IAM identity
identity = sts.get_caller_identity()
ACCOUNT_ID = identity['Account']
print(f"Account ID: {ACCOUNT_ID}")
print(f"Current identity: {identity['Arn']}")

## Section 2: Generating Synthetic PHI-Linked Genomic Data

In [ ]:
# Generate realistic but completely synthetic patient-linked genomic data
# This data is fabricated — no real patient information
fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

tissues = ['Brain', 'Liver', 'Lung', 'Heart', 'Kidney', 'Blood', 'Breast']
cancer_types = ['BRCA', 'GBM', 'LUAD', 'COAD', 'LIHC']
platforms = ['NovaSeq 6000', 'HiSeq X', 'NextSeq 550']

n_patients = 50
records = []

for i in range(n_patients):
    dob = fake.date_of_birth(minimum_age=35, maximum_age=80)
    records.append({
        # PHI columns (protected health information)
        'mrn': f'MRN-{fake.numerify("######")}',
        'patient_name': fake.name(),
        'date_of_birth': dob.strftime('%Y-%m-%d'),
        'ssn_last4': fake.numerify('####'),
        
        # Non-PHI clinical data
        'sample_id': f'SMP-{i+1:04d}',
        'tissue_type': random.choice(tissues),
        'cancer_type': random.choice(cancer_types),
        'tumor_stage': random.choice(['I', 'II', 'III', 'IV']),
        'sequencing_platform': random.choice(platforms),
        'collection_year': random.randint(2018, 2025),
        
        # Genomic results (non-PHI in aggregate)
        'pathogenic_variant_count': random.randint(0, 15),
        'vus_count': random.randint(0, 30),
        'benign_count': random.randint(50, 200),
        'tmb_per_mb': round(random.uniform(0.5, 35.0), 2),
        'msi_status': random.choice(['MSS', 'MSS', 'MSS', 'MSI-L', 'MSI-H']),
        'brca1_status': random.choice(['WT', 'WT', 'WT', 'Pathogenic', 'VUS']),
        'brca2_status': random.choice(['WT', 'WT', 'WT', 'Pathogenic', 'VUS']),
        'tp53_mutated': random.choice([True, True, False]),
    })

df_patients = pd.DataFrame(records)

print(f"Generated {len(df_patients)} synthetic patient records")
print(f"\nColumns:")
for col in df_patients.columns:
    phi = '*** PHI ***' if col in ['mrn', 'patient_name', 'date_of_birth', 'ssn_last4'] else ''
    print(f"  {col}: {df_patients[col].dtype}  {phi}")

In [ ]:
# Write to S3 and register in Glue catalog
wr.s3.to_parquet(
    df=df_patients,
    path=f"s3://{BUCKET}/silver/clinical/patient_genomics/",
    dataset=True,
    database='binfx410_silver',
    table='patient_genomics',
    partition_cols=['tissue_type'],
    boto3_session=session,
    mode='overwrite'
)

print("Patient genomics table registered:")
print(f"  s3://{BUCKET}/silver/clinical/patient_genomics/")
print(f"  Glue table: binfx410_silver.patient_genomics")
print(f"  Partitioned by: tissue_type")
print()
print("IMPORTANT: This synthetic data is FICTIONAL.")
print("Real patient data must never be used in training exercises.")

## Section 3: Registering the S3 Location with Lake Formation

In [ ]:
# Register the S3 location with Lake Formation
# This tells Lake Formation: "I control access to this S3 path"
# After registration, S3 bucket policies alone are no longer sufficient

try:
    lf.register_resource(
        ResourceArn=f'arn:aws:s3:::{BUCKET}',
        UseServiceLinkedRole=True
    )
    print(f"Registered s3://{BUCKET} with Lake Formation.")
except lf.exceptions.AlreadyExistsException:
    print(f"s3://{BUCKET} is already registered with Lake Formation.")

# Confirm registration
resources = lf.list_resources()
print(f"\nRegistered Lake Formation resources:")
for res in resources.get('ResourceInfoList', []):
    print(f"  {res['ResourceArn']}")

## Section 4: Creating IAM Roles for Two User Types

In [ ]:
# Trust policy — allows the current user to assume these roles
current_user_arn = identity['Arn']

trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"AWS": current_user_arn},
        "Action": "sts:AssumeRole"
    }]
})

# Minimum IAM policy for Athena + Lake Formation queries
athena_lf_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["athena:StartQueryExecution", "athena:GetQueryExecution",
                       "athena:GetQueryResults", "athena:StopQueryExecution"],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["glue:GetTable", "glue:GetDatabase", "glue:GetPartitions"],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["lakeformation:GetDataAccess"],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET}",
                f"arn:aws:s3:::{BUCKET}/*",
                f"arn:aws:s3:::binfx410-athena-results-{STUDENT_ID}",
                f"arn:aws:s3:::binfx410-athena-results-{STUDENT_ID}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": ["s3:PutObject"],
            "Resource": f"arn:aws:s3:::binfx410-athena-results-{STUDENT_ID}/*"
        }
    ]
})

# Create researcher role (limited access)
for role_name in ['binfx410-researcher-role', 'binfx410-clinical-role']:
    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=trust_policy,
            Description=f'BINFX410 {role_name} for Lake Formation governance exercise'
        )
        print(f"Created role: {role_name}")
    except iam.exceptions.EntityAlreadyExistsException:
        print(f"Role already exists: {role_name}")

# Attach Athena + LF access policy to both roles
for role_name in ['binfx410-researcher-role', 'binfx410-clinical-role']:
    try:
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName='AthenLakeFormationAccess',
            PolicyDocument=athena_lf_policy
        )
    except: pass

RESEARCHER_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/binfx410-researcher-role'
CLINICAL_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/binfx410-clinical-role'
print(f"\nResearcher role: {RESEARCHER_ARN}")
print(f"Clinical role:   {CLINICAL_ARN}")

## Section 5: Granting Table Permissions with Lake Formation

In [ ]:
# First: grant both roles access to the Glue database
for role_arn in [RESEARCHER_ARN, CLINICAL_ARN]:
    try:
        lf.grant_permissions(
            Principal={'DataLakePrincipalIdentifier': role_arn},
            Resource={
                'Database': {
                    'CatalogId': ACCOUNT_ID,
                    'Name': 'binfx410_silver'
                }
            },
            Permissions=['DESCRIBE']
        )
        print(f"Granted DESCRIBE on database to: {role_arn.split('/')[-1]}")
    except Exception as e:
        print(f"Note: {e}")

In [ ]:
# Grant CLINICAL ROLE: full table access (all columns, all rows)
try:
    lf.grant_permissions(
        Principal={'DataLakePrincipalIdentifier': CLINICAL_ARN},
        Resource={
            'Table': {
                'CatalogId': ACCOUNT_ID,
                'DatabaseName': 'binfx410_silver',
                'Name': 'patient_genomics'
            }
        },
        Permissions=['SELECT', 'DESCRIBE']
    )
    print("Clinical role: granted SELECT on ALL columns (full PHI access)")
except Exception as e:
    print(f"Note (may already exist): {e}")

In [ ]:
# Grant RESEARCHER ROLE: SELECT but EXCLUDE PHI columns
# ColumnWildcard with ExcludedColumnNames masks specific columns

PHI_COLUMNS = ['mrn', 'patient_name', 'date_of_birth', 'ssn_last4']

try:
    lf.grant_permissions(
        Principal={'DataLakePrincipalIdentifier': RESEARCHER_ARN},
        Resource={
            'TableWithColumns': {
                'CatalogId': ACCOUNT_ID,
                'DatabaseName': 'binfx410_silver',
                'Name': 'patient_genomics',
                'ColumnWildcard': {
                    'ExcludedColumnNames': PHI_COLUMNS
                }
            }
        },
        Permissions=['SELECT', 'DESCRIBE']
    )
    print(f"Researcher role: granted SELECT excluding PHI columns:")
    for col in PHI_COLUMNS:
        print(f"  - {col} (masked)")
except Exception as e:
    print(f"Note: {e}")

## Section 6: Applying a Row-Level Filter

In [ ]:
# Researcher role also gets a row filter: only Brain tissue samples
# This simulates a researcher approved only for the neuroscience cohort

# First create a data filter
try:
    lf.create_data_cells_filter(
        TableData={
            'TableCatalogId': ACCOUNT_ID,
            'DatabaseName': 'binfx410_silver',
            'TableName': 'patient_genomics',
            'Name': 'brain_tissue_only',
            'RowFilter': {
                'FilterExpression': "tissue_type = 'Brain'"
            },
            'ColumnWildcard': {
                'ExcludedColumnNames': PHI_COLUMNS
            }
        }
    )
    print("Created row filter: brain_tissue_only")
except Exception as e:
    print(f"Note: {e}")

# Grant the researcher role access via this filter
try:
    lf.grant_permissions(
        Principal={'DataLakePrincipalIdentifier': RESEARCHER_ARN},
        Resource={
            'DataCellsFilter': {
                'TableCatalogId': ACCOUNT_ID,
                'DatabaseName': 'binfx410_silver',
                'TableName': 'patient_genomics',
                'Name': 'brain_tissue_only'
            }
        },
        Permissions=['SELECT']
    )
    print("Researcher role granted access via brain_tissue_only filter")
except Exception as e:
    print(f"Note: {e}")

## Section 7: Testing Access Control

In [ ]:
# Test as clinical role — should see all columns and all rows

def query_as_role(role_arn, sql, database='binfx410_silver'):
    """Assume an IAM role and run an Athena query as that role."""
    # Assume the role
    assumed = sts.assume_role(
        RoleArn=role_arn,
        RoleSessionName='binfx410-governance-test'
    )
    creds = assumed['Credentials']
    
    # Create a boto3 session with the assumed role credentials
    assumed_session = boto3.Session(
        aws_access_key_id=creds['AccessKeyId'],
        aws_secret_access_key=creds['SecretAccessKey'],
        aws_session_token=creds['SessionToken'],
        region_name=AWS_REGION
    )
    
    # Query via awswrangler
    try:
        result = wr.athena.read_sql_query(
            sql=sql,
            database=database,
            s3_output=ATHENA_RESULTS,
            boto3_session=assumed_session
        )
        return result, None
    except Exception as e:
        return None, str(e)

print("Role assumption helper defined.")
print()
print("NOTE: sts:AssumeRole may take 10-30 seconds for new roles to propagate.")

In [ ]:
# Wait for IAM propagation [NOTE: This will not work if you are Root]
import time
print("Waiting 15 seconds for IAM role propagation...")
time.sleep(15)

test_query = """
SELECT *
FROM binfx410_silver.patient_genomics
LIMIT 3
"""

print("=== CLINICAL ROLE (full access) ===")
clinical_result, clinical_error = query_as_role(CLINICAL_ARN, test_query)
if clinical_result is not None:
    print(f"Columns visible ({len(clinical_result.columns)}): {list(clinical_result.columns)}")
    print(f"Rows returned: {len(clinical_result)}")
    print(clinical_result[['mrn', 'patient_name', 'sample_id', 'tissue_type', 'pathogenic_variant_count']].to_string(index=False))
else:
    print(f"Error: {clinical_error}")

In [ ]:
print("=== RESEARCHER ROLE (PHI masked, Brain only) ===")
researcher_result, researcher_error = query_as_role(RESEARCHER_ARN, test_query)
if researcher_result is not None:
    print(f"Columns visible ({len(researcher_result.columns)}): {list(researcher_result.columns)}")
    print(f"Rows returned: {len(researcher_result)} (should be Brain tissue only)")
    print(researcher_result.head().to_string(index=False))
    
    # Verify PHI columns are absent
    phi_cols_present = [c for c in PHI_COLUMNS if c in researcher_result.columns]
    if phi_cols_present:
        print(f"WARNING: PHI columns still visible: {phi_cols_present}")
    else:
        print(f"\nPHI COLUMNS MASKED: {PHI_COLUMNS} are not present in results")
    
    # Verify row filter
    if 'tissue_type' in researcher_result.columns:
        tissues_seen = researcher_result['tissue_type'].unique()
        print(f"Tissues in result: {tissues_seen} (should only be Brain)")
else:
    print(f"Error: {researcher_error}")

## Section 8: Comparing Lake Formation vs S3 Bucket Policies

In [ ]:
comparison = pd.DataFrame([
    {'Feature': 'Column-level masking', 'S3 Bucket Policy': 'No', 'Lake Formation': 'Yes'},
    {'Feature': 'Row-level filtering', 'S3 Bucket Policy': 'No', 'Lake Formation': 'Yes'},
    {'Feature': 'Table-level access control', 'S3 Bucket Policy': 'Via prefix only', 'Lake Formation': 'Yes — via Glue catalog'},
    {'Feature': 'Centralized policy management', 'S3 Bucket Policy': 'One policy per bucket', 'Lake Formation': 'One policy for all tables'},
    {'Feature': 'Works with Athena', 'S3 Bucket Policy': 'Yes (all or nothing)', 'Lake Formation': 'Yes (column/row filtered)'},
    {'Feature': 'Works with AWS Redshift Spectrum', 'S3 Bucket Policy': 'No', 'Lake Formation': 'Yes'},
    {'Feature': 'CloudTrail audit logging', 'S3 Bucket Policy': 'Object-level (S3 access logs)', 'Lake Formation': 'Table-level (GetDataAccess logged)'},
    {'Feature': 'HIPAA suitability', 'S3 Bucket Policy': 'Insufficient alone', 'Lake Formation': 'Designed for regulated data'},
    {'Feature': 'Cost', 'S3 Bucket Policy': 'Free', 'Lake Formation': 'No charge (pay for underlying)'},
    {'Feature': 'Setup complexity', 'S3 Bucket Policy': 'Low', 'Lake Formation': 'Medium (LF admin required)'},
])

comparison.set_index('Feature', inplace=True)
comparison

## Section 9: Cleanup

In [ ]:
# Cleanup code — run this at the end of the exercise

def cleanup():
    print("Cleaning up Lake Formation permissions and IAM roles...")
    
    # Revoke permissions
    for role_arn in [RESEARCHER_ARN, CLINICAL_ARN]:
        try:
            lf.revoke_permissions(
                Principal={'DataLakePrincipalIdentifier': role_arn},
                Resource={
                    'Table': {
                        'CatalogId': ACCOUNT_ID,
                        'DatabaseName': 'binfx410_silver',
                        'Name': 'patient_genomics'
                    }
                },
                Permissions=['SELECT', 'DESCRIBE']
            )
        except: pass
    
    # Delete data cells filter
    try:
        lf.delete_data_cells_filter(
            TableCatalogId=ACCOUNT_ID,
            DatabaseName='binfx410_silver',
            TableName='patient_genomics',
            Name='brain_tissue_only'
        )
        print("  Deleted data cells filter: brain_tissue_only")
    except: pass
    
    # Delete IAM role policies and roles
    for role_name in ['binfx410-researcher-role', 'binfx410-clinical-role']:
        try:
            iam.delete_role_policy(RoleName=role_name, PolicyName='AthenLakeFormationAccess')
            iam.delete_role(RoleName=role_name)
            print(f"  Deleted IAM role: {role_name}")
        except: pass
    
    print("Cleanup complete.")

print("Cleanup function defined. Run cleanup() when finished with the exercise.")
print("DO NOT run cleanup() before answering the reflection questions.")

## Exercise: YOUR CODE HERE

In [ ]:
# TASK 1: Create a third role 'binfx410-analyst-role'
# Grant access to: sample_id, tissue_type, pathogenic_variant_count, vus_count, tmb_per_mb, msi_status
# BUT NOT: cancer_type, tumor_stage, brca1_status, brca2_status (competitive research sensitivity)
# Row filter: only collection_year >= 2022

# YOUR CODE HERE

In [ ]:
# TASK 2: Verify the analyst role sees the correct columns and rows
# Run a query as the analyst role
# Print: which columns are visible, what is the min/max collection_year in the result

# YOUR CODE HERE

In [ ]:
# TASK 3: Document the HIPAA Minimum Necessary standard
# The HIPAA Minimum Necessary Rule requires covered entities to limit PHI access
# to only what is needed for a specific purpose.
# 
# Fill in this table for three researcher personas in your lab:

personas = pd.DataFrame([
    {
        'persona': 'Bioinformatician (no patient contact)',
        'needs_mrn': None,          # True or False
        'needs_dob': None,
        'needs_tissue_type': None,
        'needs_genomic_results': None,
        'row_restriction': None,    # e.g., 'own study cohort only' or 'all'
        'justification': None       # one sentence
    },
    {
        'persona': 'Clinical geneticist (returns results to patients)',
        'needs_mrn': None,
        'needs_dob': None,
        'needs_tissue_type': None,
        'needs_genomic_results': None,
        'row_restriction': None,
        'justification': None
    },
    {
        'persona': 'External collaborator (approved IRB, no PHI access)',
        'needs_mrn': None,
        'needs_dob': None,
        'needs_tissue_type': None,
        'needs_genomic_results': None,
        'row_restriction': None,
        'justification': None
    }
])

# YOUR CODE: fill in the None values above, then display
personas.set_index('persona', inplace=True)
personas

## Reflection Questions

*(Double-click to edit)*

1. **The HIPAA Privacy Rule specifies the 18 identifiers** that make health information "individually identifiable" (name, DOB, MRN, geographic data below state level, etc.). Our synthetic dataset masked name, DOB, MRN, and SSN. What other fields in the dataset could potentially be re-identifying when combined? (Think about the combination of tissue_type + cancer_type + collection_year.)

2. **dbGaP (the NIH database of Genotypes and Phenotypes)** requires researchers to submit a Data Access Request specifying exactly what data they need and why. How does Lake Formation's column- and row-level access control map to this concept? Could Lake Formation enforce a dbGaP data access agreement computationally?

3. **Lake Formation denies access at the Athena query level**, but the underlying Parquet files in S3 still exist. If a researcher with only the `researcher-role` obtains temporary AWS credentials and directly calls `s3.get_object()` on the Parquet files, will they get the PHI? What does this tell you about the threat model Lake Formation is designed to address?

4. **Clinical genomics labs often have a 'break-glass' procedure** — an emergency mechanism allowing a clinician to override access controls in an urgent patient care situation. How would you implement break-glass access in an AWS Lake Formation architecture while still maintaining an audit trail?

5. **Compare two governance architectures:**
   - Architecture A: One S3 bucket per sensitivity level (public bucket, restricted bucket, PHI bucket)
   - Architecture B: One bucket, Lake Formation column/row controls
   
   What are the operational trade-offs? Which would you recommend for a genomics lab that is growing and will add 5 new researchers next year, each needing different data access?

*(Write answers here)*

1. 

2. 

3. 

4. 

5. 

In [ ]:
cleanup()

In [ ]:
## End of Notebook - Remember to run cleanup() ##